# Data load & Sanity Check

In [3]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("processed_10")
DATA_DIR = OUTPUT_DIR / "normalized"

split = json.loads(
    (OUTPUT_DIR / "split.json").read_text()
)

def load_split(song_ids):
    Xs = []
    Ys = []

    for song_id in song_ids:
        data = np.load(DATA_DIR / song_id / "windows.npz")

        Xs.append(data["X"])
        Ys.append(data["Y"])

    return (
        np.concatenate(Xs, axis=0).astype(np.float32),
        np.concatenate(Ys, axis=0).astype(np.float32)
    )

X_train, Y_train = load_split(split["train"])
X_val, Y_val = load_split(split["validation"])
X_test, Y_test = load_split(split["test"])

print("Train:", X_train.shape, Y_train.shape)
print("Val:  ", X_val.shape, Y_val.shape)
print("Test: ", X_test.shape, Y_test.shape)

assert X_train.shape[1] == 1385
assert Y_train.shape[1:] == (4, 7)

assert X_val.shape[1] == 1385
assert Y_val.shape[1:] == (4, 7)

assert X_test.shape[1] == 1385
assert Y_test.shape[1:] == (4, 7)

Train: (41048, 1385) (41048, 4, 7)
Val:   (4915, 1385) (4915, 4, 7)
Test:  (6031, 1385) (6031, 4, 7)


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: mps


### Model Pipeline

**One input window**  
↓  
**16 audio frames (16 × 80) + 15 previous note frames (15 × 7)**  
↓  
**CNN extracts features from the audio**  
↓  
**Audio features are combined with previous note information**  
↓  
**LSTM learns temporal/rhythmic patterns**  
↓  
**Predict 4 timesteps (4 × 7 classes)**

In [ ]:
class TaikoNationBaseline(nn.Module):
    def __init__(self):
        super().__init__()

        # Split the 1385 inputs; split them back into audio and notes
        self.conv1 = nn.Conv1d(80, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(2)

        self.audio_fc = nn.Linear(32 * 4, 128)

        self.lstm1 = nn.LSTM(
            input_size=8,
            hidden_size=64,
            batch_first=True
        )

        self.lstm2 = nn.LSTM(
            input_size=64,
            hidden_size=64,
            batch_first=True
        )

        self.output = nn.Linear(64, 28)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.8)

    # send the audio through the CNN to find useful patters in the audio features
    def forward(self, x):
        audio = x[:, :1280].reshape(-1, 16, 80)
        notes = x[:, 1280:].reshape(-1, 15, 7)

        audio = audio.transpose(1, 2)

        audio = self.relu(self.conv1(audio))
        audio = self.dropout(audio)
        audio = self.pool1(audio)

        audio = self.relu(self.conv2(audio))
        audio = self.pool2(audio)

        audio = audio.flatten(1)
        audio = self.relu(self.audio_fc(audio))
        audio = audio.reshape(-1, 16, 8)

        note_bias = torch.ones(
            notes.shape[0], 15, 1,
            device=notes.device,
            dtype=notes.dtype
        )

        note_history = torch.cat([notes, note_bias], dim=2)

        current_placeholder = torch.ones(
            notes.shape[0], 1, 8,
            device=notes.device,
            dtype=notes.dtype
        )

        note_context = torch.cat(
            [note_history, current_placeholder],
            dim=1
        )

        # Combine audio and previous-note information
        combined = audio * note_context

        combined, _ = self.lstm1(combined)
        combined = self.relu(combined)
        combined = self.dropout(combined)

        # LSTM learns the temporal pattern
        combined, _ = self.lstm2(combined)
        combined = self.relu(combined)
        combined = self.dropout(combined)

        final = combined[:, -1, :]
        logits = self.output(final)

        return logits.reshape(-1, 4, 7)

# sanity check

In [6]:
model = TaikoNationBaseline().to(device)

sample = torch.from_numpy(X_train[:8]).float().to(device)

with torch.no_grad():
    output = model(sample)

print("Input:", sample.shape)
print("Output:", output.shape)

assert output.shape == (8, 4, 7)

Input: torch.Size([8, 1385])
Output: torch.Size([8, 4, 7])


# data loader

In [7]:
#data loader
train_dataset = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(Y_train).float()
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val).float(),
    torch.from_numpy(Y_val).float()
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 2566
Val batches: 308


# loss + optimizer

In [ ]:
# measures how wrong the model's predictions are compared withthe correct note classes
criterion = nn.CrossEntropyLoss()

# changes the model's weights to reduce the loss; Adam is a popular optimizer that works well in many situations
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5
)

# training & validation functions

In [ ]:
# predict -> calculate loss -> backpropagate -> update weights
def train_epoch(model, loader):
    model.train()

    total_loss = 0.0

    for X, Y in loader:
        X = X.to(device)
        Y = Y.to(device)

        # before processing the new batch, we clear the gradients from the previous batch.
        optimizer.zero_grad()

        logits = model(X)
        targets = Y.argmax(dim=2)

        loss = criterion(
            logits.reshape(-1, 7),
            targets.reshape(-1)
        )

        # learn from loss; how did each model weight contribute to the loss? how should we change the weights to reduce the loss?
        loss.backward()
        # use gradients to update the model's weights  
        optimizer.step()

        total_loss += loss.item()

    # returns the average loss across all batches in the epoch
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()

    total_loss = 0.0

    with torch.no_grad():
        for X, Y in loader:
            X = X.to(device)
            Y = Y.to(device)

            logits = model(X)
            targets = Y.argmax(dim=2)

            loss = criterion(
                logits.reshape(-1, 7),
                targets.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(loader)

# Train 10 epochs

In [10]:
# first 10 epochs
train_losses = []
val_losses = []

for epoch in range(10):
    train_loss = train_epoch(model, train_loader)
    val_loss = evaluate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 01 | Train Loss: 1.5563 | Val Loss: 0.8068
Epoch 02 | Train Loss: 0.9686 | Val Loss: 0.5228
Epoch 03 | Train Loss: 0.8012 | Val Loss: 0.4207
Epoch 04 | Train Loss: 0.7301 | Val Loss: 0.3776
Epoch 05 | Train Loss: 0.6933 | Val Loss: 0.3561
Epoch 06 | Train Loss: 0.6705 | Val Loss: 0.3429
Epoch 07 | Train Loss: 0.6515 | Val Loss: 0.3335
Epoch 08 | Train Loss: 0.6370 | Val Loss: 0.3257
Epoch 09 | Train Loss: 0.6239 | Val Loss: 0.3186
Epoch 10 | Train Loss: 0.6083 | Val Loss: 0.3117


# Second stage - fine tuning

Replicate paper's training procedure with batch size 1 and learning rate 5e-6

In [14]:
train_loader_finetune = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-6
)

for epoch in range(4):
    train_loss = train_epoch(model, train_loader_finetune)
    val_loss = evaluate(model, val_loader)

    print(
        f"Fine-tune Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Fine-tune Epoch 01 | Train Loss: 0.5886 | Val Loss: 0.2984
Fine-tune Epoch 02 | Train Loss: 0.5684 | Val Loss: 0.2911
Fine-tune Epoch 03 | Train Loss: 0.5531 | Val Loss: 0.2877
Fine-tune Epoch 04 | Train Loss: 0.5419 | Val Loss: 0.2836


# save model

In [16]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "split": split
    },
    "taikonation_smoke_test.pt"
)

print("Saved.")

Saved.


# Make predictions on the test song

### Test Prediction Pipeline

**Trained Model + Test Audio/Note Windows**  
↓  
**Model Prediction**  
↓  
**Raw Scores (Logits)**  
↓  
**Softmax**  
↓  
**Probabilities for Each of the 7 Note Classes**  
↓  
**Used for Chart Generation**

In [ ]:
model.eval()

X_test_tensor = torch.from_numpy(X_test).float().to(device)

with torch.no_grad():
    logits = model(X_test_tensor)
    probs = torch.softmax(logits, dim=2).cpu().numpy()

print(probs.shape)
# print only first window's probabilities for each of the 4 notes
print(probs[0])

(6031, 4, 7)
[[0.87155765 0.06598432 0.05381141 0.00304855 0.0012839  0.00293428
  0.00137983]
 [0.872837   0.06604914 0.0507394  0.00346189 0.00125183 0.00388732
  0.00177343]
 [0.8705854  0.0664398  0.05159726 0.00322879 0.00132113 0.00473342
  0.00209422]
 [0.8674365  0.06706983 0.05234545 0.00338172 0.00133471 0.00593029
  0.00250149]]


# Combining the overlapping predictions from different windows to produce one final 7-class probability distribution for each 23 ms timestep.

In [ ]:
num_windows = probs.shape[0]
num_timesteps = num_windows + 3

aggregated = np.zeros((num_timesteps, 7), dtype=np.float32)
counts = np.zeros(num_timesteps, dtype=np.float32)

for i in range(num_windows):
    for j in range(4):
        aggregated[i + j] += probs[i, j]
        counts[i + j] += 1

aggregated /= counts[:, None]

print("Aggregated shape:", aggregated.shape)
# sanity check: the probabilities for each timestep should sum to 1
print("Probability sums:", aggregated.sum(axis=1)[:10])

Aggregated shape: (6034, 7)
Probability sums: [0.99999994 1.         1.         1.         1.         1.0000001
 1.         0.99999994 1.         1.        ]


# Generating chart

Takes the final probability distribution for every 23 ms timestep and turns it into an actual generated chart.

In [ ]:
CLASS_NAMES = [
    "none",
    "don",
    "ka",
    "DON",
    "KA",
    "drumroll",
    "denden"
]

# random generator for sampling; 42 random seed
rng = np.random.default_rng(42)

generated = np.array([
    rng.choice(7, p=p / p.sum())
    for p in aggregated
])

generated_counts = np.bincount(generated, minlength=7)

print("Generated chart:")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:10s} {generated_counts[i]:8d}")

Generated chart:
none           5464
don             235
ka              169
DON               8
KA                6
drumroll        118
denden           34


# Target vs. Model generated

In [20]:
test_targets = Y_test.argmax(axis=2)

true_sequence = np.zeros(num_timesteps, dtype=np.int64)
true_sequence[:num_windows] = test_targets[:, 0]

for j in range(1, 4):
    true_sequence[num_windows - 1 + j] = test_targets[-1, j]

true_counts = np.bincount(true_sequence, minlength=7)

print("Class       True      Generated")

for i, name in enumerate(CLASS_NAMES):
    print(
        f"{name:10s} "
        f"{true_counts[i]:8d} "
        f"{generated_counts[i]:10d}"
    )

Class       True      Generated
none           5330       5464
don             194        235
ka              304        169
DON              11          8
KA               12          6
drumroll        183        118
denden            0         34


# Onset evaluation

Regardless of note type like Don and Ka, this measures generated notes occur near the ground-truth note timings (within +- 23ms).

In [21]:
def onset_metrics(true_seq, pred_seq, tolerance=1):
    true_onsets = np.where(true_seq != 0)[0]
    pred_onsets = np.where(pred_seq != 0)[0]

    matched_true = set()
    matches = 0

    for pred in pred_onsets:
        candidates = [
            t for t in true_onsets
            if abs(t - pred) <= tolerance and t not in matched_true
        ]

        if candidates:
            best = min(candidates, key=lambda t: abs(t - pred))
            matched_true.add(best)
            matches += 1

    precision = matches / len(pred_onsets) if len(pred_onsets) else 0
    recall = matches / len(true_onsets) if len(true_onsets) else 0

    if precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    return precision, recall, f1, matches


precision, recall, f1, matches = onset_metrics(
    true_sequence,
    generated,
    tolerance=1
)

print("True onsets:", np.sum(true_sequence != 0))
print("Generated onsets:", np.sum(generated != 0))
print("Matched onsets:", matches)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

True onsets: 704
Generated onsets: 570
Matched onsets: 249
Precision: 0.4368
Recall:    0.3537
F1:        0.3909


# Onset + type eval

In [22]:
def joint_onset_type_metrics(true_seq, pred_seq, tolerance=1):
    true_onsets = np.where(true_seq != 0)[0]
    pred_onsets = np.where(pred_seq != 0)[0]

    matched_true = set()
    matches = 0

    for pred in pred_onsets:
        candidates = [
            t for t in true_onsets
            if abs(t - pred) <= tolerance
            and true_seq[t] == pred_seq[pred]
            and t not in matched_true
        ]

        if candidates:
            best = min(candidates, key=lambda t: abs(t - pred))
            matched_true.add(best)
            matches += 1

    precision = matches / len(pred_onsets) if len(pred_onsets) else 0
    recall = matches / len(true_onsets) if len(true_onsets) else 0

    if precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    return precision, recall, f1, matches


precision_type, recall_type, f1_type, matches_type = joint_onset_type_metrics(
    true_sequence,
    generated,
    tolerance=1
)

print("Matched onset + type:", matches_type)
print(f"Precision: {precision_type:.4f}")
print(f"Recall:    {recall_type:.4f}")
print(f"F1:        {f1_type:.4f}")

Matched onset + type: 146
Precision: 0.2561
Recall:    0.2074
F1:        0.2292


In [23]:
results = {
    "onset_precision": 0.4368,
    "onset_recall": 0.3537,
    "onset_f1": 0.3909,
    "joint_precision": 0.2561,
    "joint_recall": 0.2074,
    "joint_f1": 0.2292
}

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "results": results,
        "split": split
    },
    "taikonation_smoke_test.pt"
)

print("Smoke test saved.")

Smoke test saved.
